In [0]:

import requests
import os
import re
import hashlib

from pathlib import Path
from datetime import datetime, timezone


# GitHub configuration
GITHUB_OWNER = "golireddy"
GITHUB_REPO = "group-project"
GITHUB_BRANCH = "main"

# Parent directory inside GitHub
GITHUB_SOURCE_PATH = "Ingestion_data_github"

# Destination Unity Catalog Volume
VOLUME_PATH = "/Volumes/group_project/bronze/raw_files"

# GitHub API
API_BASE = (
    f"https://api.github.com/repos/"
    f"{GITHUB_OWNER}/{GITHUB_REPO}/contents"
)


session = requests.Session()

HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2026-03-10"
}


In [0]:
def get_github_contents(path):

    # Construct GitHub API URL
    url = f"{API_BASE}/{path}"

    # Send GET request
    response = session.get(
        url,
        headers=HEADERS,
        params={
            "ref": GITHUB_BRANCH,
            "per_page": 100
        },
        timeout=30
    )

    # Check for unsuccessful requests
    response.raise_for_status()

    # Convert JSON response into Python objects
    return response.json()

In [0]:
def discover_date_folders():

    # Retrieve parent directory contents
    contents = get_github_contents(
        GITHUB_SOURCE_PATH
    )

    date_folders = []

    for item in contents:

        # Only consider directories
        if item["type"] != "dir":
            continue

        folder_name = item["name"]

        # Check date format YYYY-MM-DD
        if re.fullmatch(
            r"\d{4}-\d{2}-\d{2}",
            folder_name
        ):

            try:

                datetime.strptime(
                    folder_name,
                    "%Y-%m-%d"
                )

                date_folders.append(item)

            except ValueError:
                continue

    # Sort folders chronologically
    date_folders.sort(
        key=lambda x: x["name"]
    )

    return date_folders

In [0]:
date_folders = discover_date_folders()

for folder in date_folders:

    print(f"\nProcessing: {folder['name']}")

    files = get_github_contents(
        folder["path"]
    )

    for file in files:

        if file["type"] == "file":

            print(
                file["name"],
                file["size"]
            )

In [0]:
from pathlib import Path

downloaded = 0
skipped = 0

for folder in discover_date_folders():

    date = folder["name"]

    # Create the destination folder
    dest_dir = Path(VOLUME_PATH) / f"load_date={date}"
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Get files from GitHub
    files = get_github_contents(folder["path"])

    for file in files:

        if file["type"] != "file":
            continue

        dest_path = dest_dir / file["name"]

        # Skip files that already exist
        if dest_path.exists():
            skipped += 1
            continue

        # Download the file
        response = session.get(file["download_url"])
        response.raise_for_status()

        # Save the file
        with open(dest_path, "wb") as f:
            f.write(response.content)

        downloaded += 1

print(f"Downloaded: {downloaded}")
print(f"Skipped: {skipped}")